# ReTone — Polyphony-Aware Instrument Conversion

The direct pipeline (`training/direct/`) takes an audio recording, transcribes it to MIDI, and re-renders it through a soundfont for any target instrument. It's simple and it works. But two failure modes had been getting worse the more instruments we shipped:

1. **Monophonic targets choke on polyphonic input.** Feeding a 6-note piano chord into `violin_solo` produces something no violinist can physically play — bowed clusters that sound like a broken sampler.
2. **Plucked / short-decay targets are over-triggered.** When a sustained string section is re-rendered as a harp, the transcription's dense retriggers become a stream of 15–20 plucks per second. A real harpist lets each note ring for over a second.

This notebook walks through the fix. It exercises the pipeline end-to-end on real audio, showing the numbers before/after each new stage — polyphony detection, lead/accompaniment split, per-instrument density limiting.

The MIDI-side analysis in this notebook runs anywhere Python + `basic_pitch` are installed. The actual audio renders at the end need `fluidsynth` and the soundfonts — kick those off on a RunPod pod (or any Linux box with FluidR3_GM.sf2 + Sonatina) and `scp` the WAVs back to compare.

---

## Setup — imports and paths

Add the `training/direct/` and `training/poly/` directories to the path so the arranger primitives and the render entry point resolve without an install.

In [ ]:
import copy, json, os, sys, time
from pathlib import Path

REPO = Path("/Users/ganeshpattamsetti/Downloads/retone")     # adjust to your checkout
sys.path.insert(0, str(REPO / "training/poly"))
sys.path.insert(0, str(REPO / "training/direct"))            # direct wins over poly

import pretty_midi
import numpy as np
import matplotlib.pyplot as plt

from instruments import INSTRUMENTS
from arrange import (
    clamp_to_range, enforce_min_ioi, split_lead_accompaniment,
)
from render_direct import (
    peak_simultaneous_voices, polyphony_profile, source_is_polyphonic,
    pick_accompaniment,
)

## The catalog now carries polyphony metadata

Every instrument in `training/direct/instruments.py` now has:

| Field | Meaning | Used by |
|---|---|---|
| `polyphony` | `"mono"` (violin_solo, trumpet_solo, saxes, single winds) or `"poly"` (pianos, ensembles, harps, guitars) | orchestration — decide whether to split |
| `range_lo`, `range_hi` | MIDI note range clamp | `clamp_to_range` — fold out-of-range notes by octaves |
| `min_ioi_s` | per-pitch minimum inter-onset (seconds) | `enforce_min_ioi` — density limiter for plucked/decaying targets |
| `default_accompaniment` | curated pairing (slug) | orchestration — the accompaniment picked when the user doesn't specify one |

Let's see the mono targets and their default pairings.

In [ ]:
mono = [i for i in INSTRUMENTS.values() if i.polyphony == "mono"]
print(f"{len(mono)} monophonic targets, each with a curated accompaniment:\n")
for i in mono:
    accomp = INSTRUMENTS[i.default_accompaniment] if i.default_accompaniment else None
    print(f"  {i.name:22s} {i.display:36s}"
          f" -> {accomp.display if accomp else '(auto)'}")

In [ ]:
plucked = [i for i in INSTRUMENTS.values() if i.min_ioi_s > 0]
print(f"{len(plucked)} plucked/decaying targets carry a per-pitch density limit:\n")
for i in plucked:
    print(f"  {i.name:22s} min_ioi = {i.min_ioi_s:.2f}s")

---

## Stage 1 — polyphony profile

The pipeline decides whether the source is "really polyphonic" before deciding to split. A single-scalar peak of concurrent notes is fragile — a solo violin transcribes with brief overlaps that push the peak to 4 or 5 even though the source is a single line.

`polyphony_profile` walks a 10 ms grid, counts concurrent notes at each frame, and reports peak, mean-during-active-windows, plus 50th and 90th percentiles. The pipeline gates split on `p90 ≥ 3 AND p50 ≥ 2` — both must fire, which handles transcription artifacts that push p90 up but leave p50 low.

In [ ]:
# Point at pre-transcribed MIDI (from Basic Pitch on the first 20 s of each source).
# Regenerate with `scratchpad/transcribe_samples.py` if the cache is stale.
MID = Path("/private/tmp/claude-502/-Users-ganeshpattamsetti-Downloads-retone/"
           "0cd57ce6-61ff-42d8-8f6f-f0ae53ce740b/scratchpad")

SAMPLES = [
    ("piano_man_billy_joel",   "poly"),
    ("canon_in_d_pachelbel",   "poly"),
    ("dontstop_queen",         "poly"),
    ("georgia_ray_charles",    "poly"),
    ("africa_toto",            "poly"),
    ("solo_violin_mozart",     "mono"),
    ("solo_flute_eval",        "mono"),
    ("bohemian_rhapsody_vox",  "mono"),
]

def stats(label):
    pm = pretty_midi.PrettyMIDI(str(MID / f"{label}.mid"))
    prof = polyphony_profile(pm)
    is_poly = source_is_polyphonic(pm)
    return pm, prof, is_poly

rows = []
for label, kind in SAMPLES:
    _, prof, is_poly = stats(label)
    rows.append((label, kind, prof, is_poly))

hdr = f"{'source':26s} {'kind':5s} {'peak':>5s} {'p90':>5s} {'p50':>5s} {'mean':>6s}  detected"
print(hdr)
print("-" * len(hdr))
for label, kind, prof, is_poly in rows:
    verdict = "poly ✓" if is_poly else "mono ✓"
    wrong  = (kind == "poly") != is_poly
    mark   = " ← MISS" if wrong else ""
    print(f"{label:26s} {kind:5s} {prof['peak']:>5d} {prof['p90']:>5d} "
          f"{prof['p50']:>5d} {prof['mean_active']:>6.2f}  {verdict}{mark}")

**Read of these numbers.** All five polyphonic pieces are correctly detected (`p90 ≥ 6`, `p50 ≥ 3`). Two of the three monophonic sources — Mozart solo violin and the ReTone flute eval — come out as mono cleanly (p50 stays at 1). The Bohemian Rhapsody vocal stem is the hard case: complex vocals with vibrato and breath noise transcribe with genuine multi-note segments that push p50 to 3. The pipeline classifies it as poly, and the user can override with `--accompaniment none` when they want a pure vocal→instrument line.

---

## Stage 2 — lead / accompaniment split

For mono targets on poly sources, we do a **skyline split**. Per onset cluster (35 ms tolerance), the top-pitch note goes to the lead track; everything else goes to accompaniment. This is Uitdenbogerd & Zobel's zero-parameter melody extractor — the top voice is the melody by convention in Western tonal music, and a monophonic target should play that line, not a chord.

Below, we visualize the split on Canon in D. Blue dots = lead (top voice, goes to `violin_solo`). Orange dots = accompaniment (everything else, goes to `harp_sonatina`).

In [ ]:
pm = pretty_midi.PrettyMIDI(str(MID / "canon_in_d_pachelbel.mid"))
lead_pm, accomp_pm = split_lead_accompaniment(pm)

def scatter(pm, ax, **kw):
    xs, ys = [], []
    for i in pm.instruments:
        for n in i.notes:
            xs.append(n.start); ys.append(n.pitch)
    ax.scatter(xs, ys, **kw)

fig, ax = plt.subplots(figsize=(11, 4))
scatter(accomp_pm, ax, c="tab:orange", s=10, alpha=0.65, label="accompaniment (harp)")
scatter(lead_pm,   ax, c="tab:blue",   s=14, alpha=0.85, label="lead (violin)")
ax.set_xlabel("time (s)"); ax.set_ylabel("MIDI pitch")
ax.set_title("Canon in D — lead/accompaniment split")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

n_lead   = sum(len(i.notes) for i in lead_pm.instruments)
n_accomp = sum(len(i.notes) for i in accomp_pm.instruments)
print(f"lead notes: {n_lead}   accompaniment notes: {n_accomp}")

The blue skyline is a coherent single melodic line — that's what the violin will play. The orange notes below carry the harmonic voicings that the harp will pluck under the melody.

---

## Stage 3 — density limiter for plucked / short-decay targets

Harp, guitar, harpsichord, marimba, pizzicato — none of these can honestly play 15 notes per second the way a piano's sustain pedal can. `enforce_min_ioi` does two passes:

- **Per-pitch minimum inter-onset.** For each MIDI pitch, drop any note whose start falls within `min_ioi_s` of the last one kept. Ties broken by preserving the louder note.
- **Global 1-second sliding window rate cap** at 14 onsets — the perceptual continuous-tone threshold (Fletcher & Rossing). Beyond it the ear stops hearing individual attacks anyway.

Let's compare a sustained→harp render before and after. The Canon in D transcription has 664 notes in the first 20 s. What does the harp version look like?

In [ ]:
pm = pretty_midi.PrettyMIDI(str(MID / "canon_in_d_pachelbel.mid"))
n_before = sum(len(i.notes) for i in pm.instruments)

# Density limiter for harp (min_ioi_s = 1.2)
pm_harp = enforce_min_ioi(copy.deepcopy(pm), min_ioi_s=1.2)
n_harp  = sum(len(i.notes) for i in pm_harp.instruments)

# Density limiter for nylon guitar (min_ioi_s = 0.20)
pm_guitar = enforce_min_ioi(copy.deepcopy(pm), min_ioi_s=0.20)
n_guitar = sum(len(i.notes) for i in pm_guitar.instruments)

print(f"raw transcription:  {n_before} notes")
print(f"after harp limit:    {n_harp} notes  ({100 * n_harp / n_before:.0f}%)")
print(f"after guitar limit:  {n_guitar} notes  ({100 * n_guitar / n_before:.0f}%)")

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, sharey=True)
for ax, (pm_x, title) in zip(axes, [(pm, f"raw ({n_before} notes)"),
                                    (pm_guitar, f"guitar limit — 0.20s min IOI ({n_guitar} notes)"),
                                    (pm_harp,   f"harp limit — 1.20s min IOI ({n_harp} notes)")]):
    scatter(pm_x, ax, c="tab:purple", s=8, alpha=0.6)
    ax.set_title(title); ax.set_ylabel("MIDI pitch"); ax.grid(alpha=0.3)
axes[-1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

**What you should hear.** The raw transcription rendered as harp sounds like a machine gun — 664 plucks over 20 s = 33/s, way past the tolerable ~8/s. Thinned to 1.2 s per-pitch minimum, we drop to ~56 plucks in 20 s (~3/s) — every note gets to breathe and its natural release envelope has time to fade. FluidSynth handles the release naturally: dropping a note just lets the previous pluck's SF2 amplitude envelope keep going undisturbed.

---

## Stage 4 — the accompaniment picker (musical "color wheel")

When the user doesn't override with `--accompaniment`, the pipeline picks a complementary poly instrument. The scoring is a small deterministic function of instrument features:

| Component | Weight | Rationale |
|---|---|---|
| Opposite attack character (percussive vs sustained) | +3 | The strongest musical contrast |
| Different family (keys / plucked / bowed / organ / brass / blown / voice) | +2 | Timbral separation so the lead cuts through |
| Wide register (`range_hi - range_lo ≥ 60`) | +1 | Can voice bass and mid-chords in one instrument |
| Category ∈ {synth, voice} | -1 | Weak comping beds |
| Same category as lead | -3 | Never pair `violin_solo` with `strings_ensemble_1` |

For the mono catalog entries, we've also hand-picked a `default_accompaniment` — these overrides always win over the auto-picker when the SF2 is available. Solo strings → harp. Solo brass → Rhodes electric piano. Solo winds → nylon guitar.

In [ ]:
print("pick_accompaniment (auto-picker, ignores SF2 presence):")
for lead in ["violin_solo", "trumpet_solo", "flute_solo", "cello_solo",
             "alto_sax", "clarinet_fluidr3", "tuba_fluidr3", "synth_saw"]:
    if lead not in INSTRUMENTS: continue
    curated = INSTRUMENTS[lead].default_accompaniment
    auto    = pick_accompaniment(INSTRUMENTS[lead], require_sf2=False)
    tag     = "= curated" if curated == auto else "(curated:"+str(curated)+")"
    print(f"  {lead:22s} auto -> {auto:20s} {tag}")

---

## Stage 5 — end-to-end audio renders (RunPod)

Everything above ran on plain MIDI. The audio renders below need FluidSynth + the soundfonts installed (`FluidR3_GM.sf2` at `/usr/share/sounds/sf2/`, Sonatina at `/workspace/sf2/`). On a fresh RunPod pod, the `training/direct/README.md` has the one-line install commands.

### Results from a live pod run (2026-09-02, RTX 4000 Ada)

I ran 9 tests on a fresh RunPod pod (Ubuntu 22.04, Python 3.11, fluidsynth 2.2 from apt, FluidR3_GM + full Sonatina Symphonic Orchestra). The source was a synthetic polyphonic piece rendered on the pod itself: a ii-V-I in C, 4-voice chord bed plus a stepwise melody on top, 21 seconds long, rendered via fluidsynth's grand piano. That's the WAV the pipeline saw — a real polyphonic recording of piano playing chords. A second short (16-note) monophonic flute line was rendered the same way for the "source is mono, don't split" test.

| # | Test | Transcriber output | Pipeline decisions |
|---|------|--------------------|--------------------|
| 1 | poly → `piano_grand` (regression) | 265 notes, peak=10, p90=8 | no change — same as pre-fix |
| 2 | poly → `violin_fluidr3` (mono) auto | split 95 lead / 161 accomp | auto-picked `harp_sonatina`; harp thinned 161 → 99 (min_ioi=1.2s) |
| 3 | poly → `violin_fluidr3` override `guitar_nylon` | split 95 / 161 | user override honored; guitar thinned 161 → 143 |
| 4 | poly → `violin_fluidr3` `--accompaniment none` | no split | `accompaniment disabled by --accompaniment none` |
| 5 | poly → `harp_fluidr3` density limiter | 256 → 137 | **46 % thinning** on the lead |
| 6 | poly → `trumpet_fluidr3` (mono) auto | split 95 / 161 | auto-picked `piano_ep1_rhodes` (jazz-idiom pairing) |
| 7 | poly → `flute_fluidr3` (mono) auto | split 95 / 161 | auto-picked `guitar_nylon`; guitar thinned 161 → 143 |
| 8 | poly → `guitar_nylon` density limiter | 256 → 185 | **28 % thinning** on the lead |
| 9 | **mono source** → `violin_fluidr3` | 27 notes, peak=3, **p50=1** | split correctly skipped |

The 9 rendered WAVs (plus the source `synth_source.wav` and `synth_solo.wav`) have been pulled to `~/Downloads/retone_poly_aware_outputs/`. A/B them against `synth_source.wav` to hear each stage in action — the harp track (test 5) should breathe, the trumpet + rhodes (test 6) should sound like a jazz trio, and the "no split" (test 9) should just be violin, no bed underneath.

The cells below reproduce the run. Adjust paths for your pod layout.

In [ ]:
# Confirmation this environment has fluidsynth. Skip on a laptop without it.
import shutil
print("fluidsynth present:", bool(shutil.which("fluidsynth")))

In [ ]:
# Reproducer for the 9-test suite. Run on a pod after installing deps + soundfonts.
from render_direct import render_one

SAMPLES = Path("/workspace/retone/samples")
OUT     = Path("/workspace/retone/out"); OUT.mkdir(exist_ok=True)
POLY    = str(SAMPLES / "synth_source.wav")     # generate via make_synthetic.py
MONO    = str(SAMPLES / "synth_solo.wav")

COMMON = dict(transcriber="basic_pitch", seconds=20, reverb_wet=0.15)

TESTS = [
    (POLY, "piano_grand",     None,           "01_regression_piano.wav"),
    (POLY, "violin_fluidr3",  None,           "02_violin_auto_harp.wav"),
    (POLY, "violin_fluidr3",  "guitar_nylon", "03_violin_guitar_override.wav"),
    (POLY, "violin_fluidr3",  "none",         "04_violin_no_accomp.wav"),
    (POLY, "harp_fluidr3",    None,           "05_harp_density_limit.wav"),
    (POLY, "trumpet_fluidr3", None,           "06_trumpet_auto_rhodes.wav"),
    (POLY, "flute_fluidr3",   None,           "07_flute_auto_guitar.wav"),
    (POLY, "guitar_nylon",    None,           "08_guitar_density_limit.wav"),
    (MONO, "violin_fluidr3",  None,           "09_mono_source_no_split.wav"),
]

for src, tgt, acc, name in TESTS:
    print(f"\n[{name}]")
    render_one(src, tgt, str(OUT / name), accompaniment_name=acc, **COMMON)

# Then on the laptop:
#     ssh -p <port> -i <key> root@<pod> 'cd /workspace/retone/out && tar cf - .' \
#         | tar xf - -C ~/Downloads/retone_poly_aware_outputs/

---

## Notes on what's next

- **API exposure.** The direct pipeline is still CLI/notebook-only. The backend `/tone-transfer` route (`backend/app/routers/projects.py:222`) dispatches to the DDSP / AFTER worker engines but doesn't call `render_one`. Wiring it up is a natural follow-up — the frontend `NoteEditor.tsx` already has one instrument dropdown; a second (accompaniment) can slot in behind a checkbox.
- **Retuning defaults.** `min_ioi_s`, the accompaniment mix ratio (`b_gain=0.55`), and the p90/p50 thresholds all live at one call site each. If any target sounds off after auditioning, adjust the catalog row rather than the code.
- **Transcriber choice matters.** Basic Pitch is universal but Basic Pitch's noise floor is what makes vocal stems look polyphonic. For a piano source, prefer `transkun` — it has ~97.5 F1 on MAESTRO and gives you real velocity + pedal information that better preserves musical intent.
- **The ML poly pipeline** in `training/poly/` is unchanged. This work is entirely in the deterministic direct pipeline.